In [ ]:
!pip install streamlit pyngrok bcrypt pyjwt watchdog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.7 MB/s eta 0:00:00


In [ ]:
app_code =
import streamlit as st
import sqlite3
import bcrypt
import jwt
import datetime
import re
import time
import random
import smtplib
from email.message import EmailMessage

# ================= CONFIGURATION =================
SECRET_KEY = "super_secret_key"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# --- SMTP SETTINGS ---
SMTP_EMAIL = "testmail19068@gmail.com"
SMTP_PASSWORD = "lrws swse ykrj rnjd"

# ================= DATABASE =================
def init_db():
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS users ("
              "id INTEGER PRIMARY KEY AUTOINCREMENT,"
              "username TEXT UNIQUE,"
              "email TEXT UNIQUE,"
              "password BLOB,"
              "security_question TEXT,"
              "security_answer BLOB)")
    conn.commit()
    conn.close()

init_db()

# ================= SECURITY & EMAIL =================
def send_otp_email(receiver_email, otp):
    msg = EmailMessage()
    msg["Subject"] = "Verification Code - PolicyNav"
    msg["From"] = SMTP_EMAIL
    msg["To"] = receiver_email

    html_content = f'''
    <html>
        <body style="font-family: Arial, sans-serif; background-color: #212529; padding: 20px;">
            <div style="max-width: 500px; margin: auto; background: #2b2f33; padding: 30px; border-radius: 10px; border: 2px solid #00f2ff; color: white;">
                <h2 style="color: #00f2ff; text-align: center;">PolicyNav</h2>
                <p style="text-align: center;">Your security verification code is:</p>
                <div style="background: #1a1d20; color: #ff007f; font-size: 32px; font-weight: bold; text-align: center; padding: 20px; border-radius: 5px; letter-spacing: 5px; margin: 20px 0; border: 1px solid #ff007f;">
                    {otp}
                </div>
                <p style="color: #6c757d; font-size: 12px; text-align: center;">If you did not request this, please ignore this email.</p>
            </div>
        </body>
    </html>
    '''
    msg.add_alternative(html_content, subtype='html')

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
            smtp.login(SMTP_EMAIL, SMTP_PASSWORD)
            smtp.send_message(msg)
        return True
    except: return False

def verify_security_ans_db(email, provided_answer):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("SELECT security_answer FROM users WHERE email=?", (email,))
    result = c.fetchone()
    conn.close()
    if result:
        return bcrypt.checkpw(provided_answer.lower().strip().encode(), result[0])
    return False

def get_user_question(email):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("SELECT security_question FROM users WHERE email=?", (email,))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

def email_exists(email):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email=?", (email.lower().strip(),))
    result = c.fetchone()
    conn.close()
    return result is not None

# ================= AUTH HELPERS =================
def create_access_token(data):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try: return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except: return None

def create_user(username, email, password, question, answer):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    hashed_p = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
    hashed_a = bcrypt.hashpw(answer.lower().strip().encode(), bcrypt.gensalt())
    try:
        c.execute("INSERT INTO users (username, email, password, security_question, security_answer) VALUES (?,?,?,?,?)",
                  (username, email.lower().strip(), hashed_p, question, hashed_a))
        conn.commit()
        return True
    except: return False
    finally: conn.close()

def authenticate_user(email, password):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    c.execute("SELECT username, password FROM users WHERE email=?", (email.lower().strip(),))
    user = c.fetchone()
    conn.close()
    if user and bcrypt.checkpw(password.encode(), user[1]): return user[0]
    return None

def update_password(email, new_password):
    conn = sqlite3.connect("users.db")
    c = conn.cursor()
    hashed = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt())
    c.execute("UPDATE users SET password=? WHERE email=?", (hashed, email.lower().strip()))
    conn.commit()
    conn.close()

# ================= SESSION STATE =================
if "page" not in st.session_state: st.session_state.page = "login"
if "token" not in st.session_state: st.session_state.token = None
if "reset_step" not in st.session_state: st.session_state.reset_step = 1

st.set_page_config(page_title="PolicyNav", layout="centered")

# ================= NEON UI CSS =================
st.markdown(\"\"\"
<style>
    .stApp { background-color: #212529; }

    /* PolicyNav Branding UI */
    .brand-header {
        text-align: center;
        margin-bottom: 20px;
    }
    .brand-text {
        font-size: 42px;
        font-weight: 800;
        color: #00f2ff;
        text-transform: uppercase;
        letter-spacing: 4px;
        text-shadow: 0 0 10px rgba(0, 242, 255, 0.7), 0 0 20px rgba(0, 242, 255, 0.5);
    }

    .main-card {
        background-color: #2b2f33;
        border-radius: 40px 10px 40px 10px;
        max-width: 450px;
        margin: auto;
        border-top: 2px solid #00f2ff;
        border-right: 2px solid #00f2ff;
        border-bottom: 2px solid #ff007f;
        border-left: 2px solid #ff007f;
        box-shadow: 0 0 25px rgba(0, 0, 0, 0.7);
        overflow: hidden;
    }

    .tab-container {
        display: flex;
        width: 100%;
        background-color: #1a1d20;
        border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    }

    .tab-item {
        flex: 1;
        padding: 15px 0;
        text-align: center;
        font-weight: bold;
        font-size: 14px;
        text-transform: uppercase;
        letter-spacing: 1px;
    }

    .active-tab { background-color: #2b2f33; color: #00f2ff; }
    .inactive-tab { background-color: #1a1d20; color: #6c757d; }

    .form-content { padding: 40px; }

    .stTextInput>div>div>input { background-color: #495057 !important; color: white !important; border-radius: 12px !important; border: none !important; }
    .stSelectbox>div>div { background-color: #495057 !important; color: white !important; border-radius: 12px !important; }

    .stButton>button {
        width: 100%;
        border-radius: 20px;
        height: 48px;
        background-color: #00f2ff !important;
        color: #212529 !important;
        font-weight: bold;
        border: none;
    }

    #MainMenu, footer, header {visibility: hidden;}
</style>
\"\"\", unsafe_allow_html=True)

def render_brand():
    st.markdown('<div class="brand-header"><span class="brand-text">PolicyNav</span></div>', unsafe_allow_html=True)

def render_tab_header(active):
    l = "active-tab" if active == "login" else "inactive-tab"
    s = "active-tab" if active == "signup" else "inactive-tab"
    r = "active-tab" if active == "forgot" else "inactive-tab"
    st.markdown(f'''<div class="tab-container"><div class="tab-item {l}">Sign In</div><div class="tab-item {s}">Sign Up</div><div class="tab-item {r}">Reset</div></div>''', unsafe_allow_html=True)

# ================= PAGES =================
def login_page():
    render_brand()

    render_tab_header("login")
    st.markdown('<div class="form-content">', unsafe_allow_html=True)

    e = st.text_input("Email", placeholder="yourname@gmail.com")
    p = st.text_input("Password", type="password", placeholder="••••••••")

    if st.button("LOGIN"):
        user = authenticate_user(e, p)
        if user:
            st.session_state.token = create_access_token({"username": user})
            st.rerun()
        else: st.error("Invalid Credentials")

    st.write("---")
    col1, col2 = st.columns(2)
    with col1:
        if st.button("Sign Up"):
            st.session_state.page = "signup"; st.rerun()
    with col2:
        if st.button("Forgot Password"):
            st.session_state.page = "forgot"; st.rerun()
    st.markdown('</div></div>', unsafe_allow_html=True)

def signup_page():
    render_brand()

    render_tab_header("signup")
    st.markdown('<div class="form-content">', unsafe_allow_html=True)
    u = st.text_input("Username")
    e = st.text_input("Email")
    p = st.text_input("Password", type="password")
    cp = st.text_input("Confirm Password", type="password")
    q = st.selectbox("Security Question", ["What is your pet's name?", "Mother's maiden name?", "Birth city?"])
    a = st.text_input("Security Answer")
    if st.button("REGISTER"):
        if p == cp and u and e and a:
            if create_user(u, e, p, q, a):
                st.success("Registered!"); time.sleep(1); st.session_state.page = "login"; st.rerun()
            else: st.error("User exists")
        else: st.error("Check details")
    if st.button("Cancel"):
        st.session_state.page = "login"; st.rerun()
    st.markdown('</div></div>', unsafe_allow_html=True)

def forgot_page():
    render_brand()

    render_tab_header("forgot")
    st.markdown('<div class="form-content">', unsafe_allow_html=True)
    if st.session_state.reset_step == 1:
        e = st.text_input("Enter Registered Email")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Send OTP"):
                if email_exists(e):
                    otp = str(random.randint(100000, 999999))
                    if send_otp_email(e, otp):
                        st.session_state.gen_otp = otp; st.session_state.target_email = e; st.session_state.reset_step = 2; st.rerun()
                    else: st.error("Mail error")
                else: st.error("Email not registered")
        with c2:
            if st.button("Try Another Way"):
                if email_exists(e):
                    q = get_user_question(e)
                    if q:
                        st.session_state.user_q = q; st.session_state.target_email = e; st.session_state.reset_step = 3; st.rerun()
                    else: st.error("Security not set")
                else: st.error("Email not registered")
    elif st.session_state.reset_step == 2:
        u_otp = st.text_input("Enter OTP from Email")
        if st.button("Verify OTP"):
            if u_otp == st.session_state.gen_otp: st.session_state.reset_step = 4; st.rerun()
            else: st.error("Wrong OTP")
    elif st.session_state.reset_step == 3:
        st.write(f"**Question:** {st.session_state.user_q}")
        ans = st.text_input("Your Answer")
        if st.button("Verify Answer"):
            if verify_security_ans_db(st.session_state.target_email, ans): st.session_state.reset_step = 4; st.rerun()
            else: st.error("Incorrect")
    elif st.session_state.reset_step == 4:
        np = st.text_input("New Password", type="password")
        ncp = st.text_input("Confirm", type="password")
        if st.button("Update Password"):
            if np == ncp and np: update_password(st.session_state.target_email, np); st.success("Updated!"); time.sleep(1); st.session_state.page = "login"; st.rerun()
            else: st.error("Mismatch")
    if st.button("Cancel"):
        st.session_state.reset_step = 1; st.session_state.page = "login"; st.rerun()
    st.markdown('</div></div>', unsafe_allow_html=True)

if st.session_state.token:
    data = verify_token(st.session_state.token)
    render_brand()

    st.markdown(f'<h3 style="color:#00f2ff;text-align:center;">Welcome, {data["username"]}</h3>', unsafe_allow_html=True)
    st.write("PolicyNav Dashboard")
    if st.button("LOGOUT"): st.session_state.token = None; st.rerun()
    st.markdown('</div></div>', unsafe_allow_html=True)
else:
    if st.session_state.page == "login": login_page()
    elif st.session_state.page == "signup": signup_page()
    elif st.session_state.page == "forgot": forgot_page()


with open("app.py", "w") as f:
    f.write(app_code)

In [ ]:
!pkill -f streamlit
!pkill -f ngrok

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true > log.txt 2>&1 &

In [ ]:
!tail log.txt

In [ ]:
from pyngrok import ngrok
import time

# Kill any previous ngrok sessions
ngrok.kill()

# Ask user to input token securely
authtoken = input("Enter your Ngrok Auth Token: ").strip()

if authtoken:
    ngrok.set_auth_token(authtoken)

    print("Starting Ngrok tunnel...")
    time.sleep(3)

    public_url = ngrok.connect(8501)
    print("Your Public URL:", public_url)
else:
    print("Auth token is required.")



Enter your Ngrok Auth Token: 39dxHTZwUmxQqb3b8wbivHYiFwK_4RUu8toBq7ouF6RqviS8x
Starting Ngrok tunnel...
Your Public URL: NgrokTunnel: "https://flashy-desiree-churchless.ngrok-free.dev" -> "http://localhost:8501"
